# LASPATED Discretization Demo

Interactive notebook version of `demo_discretization.py`. Use it to choose the region definition and the geographic discretization, visualize the result, and write the output files.

In [ ]:
from pathlib import Path
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import laspated as spated

DEMO_DIR = Path.cwd().resolve()
if DEMO_DIR.name != 'demo':
    candidate = DEMO_DIR / 'demo'
    if candidate.exists():
        DEMO_DIR = candidate
os.chdir(DEMO_DIR)
print('Working directory:', DEMO_DIR)

In [ ]:
region_type = 'custom'  # choose from: rectangle, convex, custom
disc_type = 'hexagon'   # choose from: rectangle, hexagon, custom

assert region_type in {'rectangle', 'convex', 'custom'}
assert disc_type in {'rectangle', 'hexagon', 'custom'}

OUTPUT_DIR = DEMO_DIR / 'disc_data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUTPUT_DIR)

In [ ]:
# Load events and initialize the data aggregator
app = spated.DataAggregator(crs='epsg:4326')
events = pd.read_csv('sorted_events.csv', encoding='ISO-8859-1', sep=',')
events.head()

In [ ]:
# Add event data
app.add_events_data(
    events,
    datetime_col='data_hora',
    lat_col='lat',
    lon_col='long',
    feature_cols=['prioridade'],
    datetime_format='%m/%d/%y %H:%M:%S',
)
print('Events loaded:', len(app.events_data))

In [ ]:
# Define the study region
if region_type == 'custom':
    max_borders = gpd.read_file('rj/rj.shp')
    app.add_max_borders(max_borders)
else:
    app.add_max_borders(method=region_type)
app.max_borders

In [ ]:
# Add time discretizations
app.add_time_discretization('m', 30, 60 * 24, column_name='hhs')
app.add_time_discretization('D', 1, 7, column_name='dow')

time_disc_df = pd.DataFrame([
    ['2016-01-01', '2016-01-01', 1, 'yearly'],
    ['2016-02-06', '2016-02-11', 2, None],
    ['2017-02-24', '2017-03-06', 2, None],
], columns=['start', 'end', 't', 'repetition'])

app.add_time_discretization(time_disc_df)
app.events_data.head()

In [ ]:
# Add the geographic discretization
if disc_type == 'rectangle':
    app.add_geo_discretization(discr_type='R', rect_discr_param_x=10, rect_discr_param_y=10)
elif disc_type == 'hexagon':
    app.add_geo_discretization(discr_type='H', hex_discr_param=7)
else:
    custom_map = gpd.read_file('rio_de_janeiro_neighborhoods/rio_neighborhoods.shp')
    app.add_geo_discretization('C', custom_data=custom_map)
print('Regions:', int(np.max(app.events_data['gdiscr']) + 1))

In [ ]:
# Plot the discretization inline
app.plot_discretization()

In [ ]:
# Add geographic features
population = gpd.read_file('populacao/')
population = population[['population', 'geometry']].copy()
app.add_geo_variable(population)
print('Geo features:', app.geo_features)

In [ ]:
# Write output files
app.write_arrivals(str(OUTPUT_DIR / 'arrivals.dat'))
app.write_regions(str(OUTPUT_DIR / 'neighbors.dat'))
app.write_info(obs_index_column='dow', path=str(OUTPUT_DIR / 'info.dat'))
plot_path = OUTPUT_DIR / 'discretization_plot.png'
app.plot_discretization(to_file=str(plot_path))
print('Wrote:', OUTPUT_DIR / 'arrivals.dat')
print('Wrote:', OUTPUT_DIR / 'neighbors.dat')
print('Wrote:', OUTPUT_DIR / 'info.dat')
print('Wrote:', plot_path)

In [ ]:
# Inspect the generated files
sorted(path.name for path in OUTPUT_DIR.iterdir())